# Exercise 1: Prompt Chaining for a Customer Support AI

**Goal:** simulate a customer service flow where each prompt's output becomes the next prompt's input.

**Tools:** Google Colab, Python 3, Google Gemini API (`gemini-2.5-flash`) via the `google-genai` SDK.

**The chain (4 steps):**

| Step | What it does | Input | Output |
|---|---|---|---|
| 1. Classify | Labels the ticket by category, urgency, sentiment | Raw ticket | JSON |
| 2. Gather missing info | Lists what we know and what we still need, writes clarifying questions | Ticket + Step 1 JSON | JSON |
| 3. Propose solution | Writes the customer reply using company policy | Ticket + Step 1 + Step 2 | Customer-facing text |
| 4. Escalation rule | Decides whether a human needs to take over, and which team | Step 1 + Step 2 + Step 3 | JSON |

## Setup

This notebook uses **Google Gemini** through the `google-genai` SDK (free tier works fine).

1. Get a free API key at https://aistudio.google.com/apikey
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GOOGLE_API_KEY`, paste your key, and turn on notebook access.
3. Run all cells top to bottom (Runtime > Run all).

If you'd rather use OpenAI, swap the body of `ask()` below. Nothing else in the notebook depends on the provider.

In [ ]:
!pip -q install -U google-genai

In [ ]:
import json, re, textwrap
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-2.5-flash"   # change here if Google renames/retires the model

def ask(prompt, system=None, temperature=0.2, json_mode=False):
    """Send one prompt to the model and return the text reply."""
    cfg = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if json_mode else "text/plain",
    )
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
    return resp.text

def ask_json(prompt, system=None, temperature=0.0):
    """Same as ask(), but parses the reply as JSON (strips ``` fences just in case)."""
    raw = ask(prompt, system=system, temperature=temperature, json_mode=True)
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    return json.loads(raw)

def show(title, obj):
    print(f"\n===== {title} =====")
    print(json.dumps(obj, indent=2) if isinstance(obj, (dict, list)) else obj)

print("Connected. Test reply:", ask("Reply with the single word: ready"))

## Company context and sample tickets

The policy text grounds Step 3 so the model can't invent refunds or timelines. The three tickets cover different paths through the chain: a clear billing issue, a vague technical issue with missing details, and an angry customer who should be escalated.

In [ ]:
POLICY = """
Acme Cloud Storage - Support Policy (internal)
- Duplicate charges: refunded in full within 5-7 business days once the charge is confirmed.
- Refunds over $100 require approval from the Billing team.
- Password resets: customer uses the "Forgot password" link; support cannot see or set passwords.
- Account lockouts after 5 failed logins last 30 minutes.
- Data loss or suspected security breach: always escalate to Tier 2 Security immediately.
- Legal threats or chargeback threats: escalate to the Account Management team.
- Support hours: Mon-Fri, 8am-6pm PT. Target first response: 4 business hours.
"""

TICKETS = {
    "T1_billing": "Hi, I was charged twice for my Pro plan this month ($24.99 each). Order #A-55812. Can you fix this?",
    "T2_vague_tech": "your app isnt working. files wont upload. been like this for days. fix it",
    "T3_angry_escalate": ("This is the third time I'm writing. Half my team's shared folder disappeared overnight "
                          "and nobody has answered me. We're a 40-person company on the Business plan. "
                          "If this isn't fixed today I'm disputing the charge with my bank and talking to a lawyer."),
}

## Iteration: Step 1 before vs. after

My first classifier prompt was one line. Running it showed two problems: the output format changed from ticket to ticket (sometimes a sentence, sometimes a list), and it gave no urgency, so Step 4 had nothing solid to base escalation on. You can't reliably feed free text into the next step.

**v1 prompt (before):**

In [ ]:
V1_CLASSIFY = "What kind of problem is this customer having?\n\nTicket: {ticket}"

for tid, t in TICKETS.items():
    show(f"v1 classifier on {tid}", ask(V1_CLASSIFY.format(ticket=t)))

**What I changed for v2:** gave the model a role, a fixed list of allowed categories, an urgency scale with definitions, and required JSON with exact keys. I also told it not to guess details that aren't in the ticket. Now Steps 2 to 4 can read specific fields (`category`, `urgency`) instead of parsing prose.

*(Add a sentence or two here about what you saw in the v1 outputs above after you run it.)*

## The chain prompts (v2, final)

In [ ]:
STEP1_SYSTEM = """You are a support ticket triage specialist for Acme Cloud Storage.
Classify the ticket. Do not guess facts that are not in the ticket.
Return ONLY JSON with exactly these keys:
{
  "category": one of ["billing", "technical", "account_access", "data_loss", "other"],
  "urgency": one of ["low", "medium", "high"],
  "sentiment": one of ["positive", "neutral", "frustrated", "angry"],
  "one_line_summary": string, max 20 words
}
Urgency guide: high = data loss, security, legal/chargeback threat, or business is blocked;
medium = a paid feature is broken or money is wrong; low = questions and minor issues."""

STEP2_SYSTEM = """You are a support agent preparing to reply to a ticket.
You receive the original ticket and the triage JSON from the previous step.
Use the triage category to decide what details a support agent would need for that kind of issue
(example: billing needs order number and amount; technical needs device/OS, file type/size, error message).
Return ONLY JSON with exactly these keys:
{
  "known_facts": list of facts stated in the ticket,
  "missing_info": list of details still needed to resolve it (empty list if none),
  "clarifying_questions": list of at most 3 short, plain-language questions that ask for the missing_info
}
Never ask for a password, full card number, or other sensitive credentials."""

STEP3_SYSTEM = """You are a friendly, professional Acme Cloud Storage support agent writing the reply email.
You receive: the ticket, the triage JSON (step 1), and the missing-info JSON (step 2), plus company policy.
Rules:
- Max 150 words. Plain language, no jargon, no corporate filler.
- If sentiment is frustrated or angry, acknowledge it in one sentence without over-apologizing.
- Only promise what the POLICY allows. Never invent timelines, credits, or refunds.
- If step 2 has clarifying_questions, include them as a short numbered list.
- If the issue matches an escalation case in the policy, say a specialist team will follow up; do not try to solve it yourself.
- Sign off as "Acme Support". Output only the email body."""

STEP4_SYSTEM = """You are a support team lead deciding whether a ticket needs a human specialist.
You receive the triage JSON, the missing-info JSON, and the drafted reply.
Escalation rules (apply in order, first match wins):
1. category is data_loss OR the ticket mentions a security breach -> route_to "tier2_security"
2. ticket mentions a lawyer, legal action, chargeback, or disputing a charge -> route_to "account_management"
3. a refund over $100 is involved -> route_to "billing_team"
4. urgency is high for any other reason -> route_to "tier2_support"
5. otherwise -> escalate false, route_to "none"
Also check the drafted reply: if it promises anything the rules or policy don't allow, set "reply_needs_edit": true.
Return ONLY JSON with exactly these keys:
{"escalate": bool, "route_to": string, "rule_matched": int, "reason": string (max 25 words), "reply_needs_edit": bool}"""

## Running the chain

In [ ]:
def run_chain(ticket):
    # Step 1: classify
    s1 = ask_json(f"Ticket:\n{ticket}", system=STEP1_SYSTEM)

    # Step 2: uses the ticket + Step 1 output
    s2 = ask_json(
        f"Ticket:\n{ticket}\n\nTriage JSON (step 1):\n{json.dumps(s1)}",
        system=STEP2_SYSTEM)

    # Step 3: uses ticket + Step 1 + Step 2 + policy
    s3 = ask(
        f"POLICY:\n{POLICY}\n\nTicket:\n{ticket}\n\n"
        f"Triage JSON (step 1):\n{json.dumps(s1)}\n\n"
        f"Missing info JSON (step 2):\n{json.dumps(s2)}",
        system=STEP3_SYSTEM, temperature=0.4)

    # Step 4: uses Steps 1-3
    s4 = ask_json(
        f"Ticket:\n{ticket}\n\nTriage JSON (step 1):\n{json.dumps(s1)}\n\n"
        f"Missing info JSON (step 2):\n{json.dumps(s2)}\n\nDrafted reply (step 3):\n{s3}",
        system=STEP4_SYSTEM)

    return {"step1_classify": s1, "step2_missing_info": s2, "step3_reply": s3, "step4_escalation": s4}

results = {}
for tid, t in TICKETS.items():
    print("\n" + "#" * 70 + f"\nTICKET {tid}\n" + "#" * 70)
    print(t)
    results[tid] = run_chain(t)
    for k, v in results[tid].items():
        show(k, v)

## Quick sanity checks on the chain output

In [ ]:
for tid, r in results.items():
    words = len(r["step3_reply"].split())
    print(f"{tid}: category={r['step1_classify']['category']:<15} urgency={r['step1_classify']['urgency']:<7} "
          f"questions={len(r['step2_missing_info']['clarifying_questions'])}  reply_words={words:<4} "
          f"escalate={r['step4_escalation']['escalate']} -> {r['step4_escalation']['route_to']}")

# What I expect: T1 no escalation (refund is under $100), T2 asks clarifying questions,
# T3 escalates (data loss rule fires first).
assert all(len(r["step3_reply"].split()) <= 170 for r in results.values()), "a reply ran long"
assert results["T3_angry_escalate"]["step4_escalation"]["escalate"] is True
assert len(results["T2_vague_tech"]["step2_missing_info"]["clarifying_questions"]) >= 1
print("\nChecks passed.")

## Notes

- **How each step uses prior output:** Step 2 picks which details to ask for based on Step 1's `category`. Step 3 changes tone based on Step 1's `sentiment` and pulls its questions straight from Step 2. Step 4 applies escalation rules to Step 1's `category`/`urgency` and audits the Step 3 draft.
- **Constraints used:** fixed category lists, JSON-only output with exact keys, 150-word limit, policy grounding, no requests for passwords or card numbers, ordered escalation rules.
- *(Add anything you tweaked while testing, e.g. if a reply ran long or the escalation picked the wrong team.)*